In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import warnings
warnings.filterwarnings("ignore")

In [4]:
chain_df = pd.read_csv('spx_w/clean_options_chain.csv', header=0)
with open("kalshi_data.pkl", "rb") as f:
    kalshi_df = pickle.load(f)

In [5]:
chain_df[["quote", "exp"]] = chain_df[["quote", "exp"]].apply(pd.to_datetime, format='%Y-%m-%d')

In [92]:
pmf_df

,3200.0-3399.99,3400.0-3599.99,3600.0-3799.99,3800.0-3999.99,4000.0-4199.99,4200.0-4399.99,4400.0-4599.99,4600.0-4799.99,4800.0-4999.99,5000.0-5199.99,5200.0-5399.99,5400.0-5599.99,5600.0-5799.99
2024-01-01,0.020115,0.034366,0.049925,0.023140,0.004526,0.042449,0.081904,0.075841,0.083045,0.129825,0.144820,0.106291,0.061931
2024-01-02,0.020115,0.034366,0.049925,0.023140,0.004526,0.042449,0.081904,0.075841,0.083045,0.129825,0.144820,0.106291,0.061931
2024-01-03,0.011885,0.042413,0.069494,0.061955,0.001509,0.012696,0.080422,0.111309,0.105610,0.101971,0.125741,0.117456,0.063589
2024-01-04,0.022304,0.046096,0.098032,0.005799,0.003373,0.092765,0.110657,0.053908,0.068019,0.147578,0.151829,0.090264,0.048013
2024-01-05,0.020876,0.028244,0.083007,0.088439,0.000587,0.005214,0.102607,0.144740,0.107041,0.078540,0.123444,0.125989,0.063219
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-27,0.001091,0.001700,0.001140,0.001097,0.000527,0.022635,0.028276,0.029361,0.044057,0.015623,0.017551,0.054178,0.151514
2024-12-28,0.001091,0.001700,0.001140,0.001097,0.000527,0.022635,0.028276,0.029361,0.044057,0.015623,0.017551,0.054178,0.151514
2024-12-29,0.001091,0.001700,0.001140,0.001097,0.000527,0.022635,0.028276,0.029361,0.044057,0.015623,0.017551,0.054178,0.151514
2024-12-30,0.001091,0.001700,0.001140,0.001097,0.000527,0.022635,0.028276,0.029361,0.044057,0.015623,0.017551,0.054178,0.151514


In [93]:
bucket_edges

array([[3200.  , 3399.99],
       [3400.  , 3599.99],
       [3600.  , 3799.99],
       [3800.  , 3999.99],
       [4000.  , 4199.99],
       [4200.  , 4399.99],
       [4400.  , 4599.99],
       [4600.  , 4799.99],
       [4800.  , 4999.99],
       [5000.  , 5199.99],
       [5200.  , 5399.99],
       [5400.  , 5599.99],
       [5600.  , 5799.99]])

In [87]:
pmf_breeden = {}

for year in kalshi_df.keys():
    # Pre-allocate an empty DataFrame matching Kalshi's structure
    pmf_df = kalshi_df[year]["price"].copy()
    pmf_df[:] = np.nan
    pmf_df.index = pd.to_datetime(pmf_df.index)

    # Prepare the chain for the year — only calls
    chain_temp = chain_df[
        (chain_df["quote"].dt.year == year) & (chain_df["option_type"] == "c")
    ][["quote", "strike", "BSM", "fwd"]].copy()

    # Precompute the forward curve
    spx_fwd = chain_df.groupby("quote")["fwd"].first()
    spx_fwd = spx_fwd.reindex(pmf_df.index, method="ffill").bfill()

    # Precompute bucket edges
    bucket_edges = np.array([
        list(map(lambda x: float(x), bucket.split('-')))
        for bucket in pmf_df.columns
    ])
    bucket_lowers = bucket_edges[:, 0]
    bucket_uppers = bucket_edges[:, 1]

    # Process each date
    for date in pmf_df.index:
        calls = chain_temp[chain_temp["quote"] == date]

        # Skip if not enough strikes
        if calls.empty or calls.shape[0] < 5:
            continue

        # Sort by strike
        calls = calls.sort_values("strike")
        strikes = calls["strike"].values
        call_prices = calls["BSM"].values

        # First and second derivatives
        dC_dK = np.gradient(call_prices, strikes)
        d2C_dK2 = np.gradient(dC_dK, strikes)

        # Risk-neutral PDF is second derivative
        pdf_vals = d2C_dK2

        # Clean up minor negatives due to noise
        pdf_vals = np.clip(pdf_vals, 0, None)

        # Interpolate PDF across a fine grid for integration
        fine_strikes = np.linspace(strikes.min(), strikes.max(), 1000)
        fine_pdf = np.interp(fine_strikes, strikes, pdf_vals, left=0, right=0)

        # Integrate PDF over each bucket
        bucket_probs = []
        for L, U in zip(bucket_lowers, bucket_uppers):
            # Select fine strikes within bucket
            mask = (fine_strikes >= L) & (fine_strikes <= U)
            if np.any(mask):
                prob = np.trapz(fine_pdf[mask], fine_strikes[mask])
            else:
                prob = 0.0
            bucket_probs.append(prob)

        # Set the probs for this date
        pmf_df.loc[date, :] = bucket_probs

    # Final clean-up: fill missing dates
    pmf_df = pmf_df.fillna(method="ffill").fillna(method="bfill")

    # Store
    pmf_breeden[year] = pmf_df

In [ ]:
opt_price_bucket_probs = {}

for year in kalshi_df.keys():
    temp = kalshi_df[year]["price"].copy()
    temp[:] = np.nan
    temp.index = pd.to_datetime(temp.index)
    chain_temp = chain_df.loc[(chain_df["quote"].dt.year == year) & (chain_df["option_type"] == "c"), ["quote", "strike", "BSM"]].copy()
    chain_temp.set_index("quote", inplace=True)
    spx_fwd = chain_temp.groupby("quote")["fwd"].first()
    spx_fwd = pd.DataFrame(spx_fwd.reindex(temp.index, method="ffill").bfill())
    M_range = [np.mean(tuple(map(lambda x : round(float(x)), bucket.split('-')))) for bucket in temp.columns]

    d_max = pd.DataFrame(index=spx_fwd.index, columns=["d_max"])
    d_max["d_max_left"] = np.abs(min(M_range) - spx_fwd["fwd"])
    d_max["d_max_right"] = np.abs(max(M_range) - spx_fwd["fwd"])
    d_max = d_max[["d_max_left", "d_max_right"]].max(axis=1)
    for bucket in temp.columns:
        L, U = map(lambda x : round(float(x)), bucket.split('-'))
        # For calls, p = (c(L) - c(U)) / (U - L) and for puts, p = (p(U) - p(L)) / (U - L)
        calls_p = (chain_temp[(chain_temp["option_type"] == "c") & (chain_temp["strike"] == L)]["BSM"] - 
                                chain_temp[(chain_temp["option_type"] == "c") & (chain_temp["strike"] == U)]["BSM"]) / (U - L)
        puts_p = (chain_temp[(chain_temp["option_type"] == "p") & (chain_temp["strike"] == U)]["BSM"] -
                                chain_temp[(chain_temp["option_type"] == "p") & (chain_temp["strike"] == L)]["BSM"]) / (U - L)
        calls_p = calls_p.reindex(spx_fwd.index, method="ffill").bfill()
        puts_p = puts_p.reindex(spx_fwd.index, method="ffill").bfill()
        M = (L + U) / 2
        w_puts = 0.5 * (1 - (M - spx_fwd["fwd"]) / d_max)
        w_puts = np.clip(w_puts, 0, 1)
        w_calls = 1 - w_puts
        temp[bucket] = w_calls * calls_p + w_puts * puts_p
    # Round and normalize the probabilities
    temp = temp.div(temp.sum(axis=1), axis=0).round(4)
    opt_price_bucket_probs[year] = temp.copy()

In [78]:
sample_date = "2022-11-10"

temp_optprob = opt_price_bucket_probs[int(sample_date[:4])].loc[sample_date].copy()
temp_kalshiprob = kalshi_df[int(sample_date[:4])]["price"].loc[sample_date].copy()


